# TULIP: self-contained example

This notebook uses the bundled mock data in `data/mock_tulip_data.npz`. It creates a synthetic mixed image from a labelled cell mask, then recovers the spectra with non-negative least squares. No private data or absolute paths are required.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from tulip import MIX, NNLS, SPATIAL, SPECTRAL

data_path = Path("data/mock_tulip_data.npz")
if not data_path.exists():
    data_path = Path("example/data/mock_tulip_data.npz")
data = np.load(data_path)
np.random.seed(0)  # Reproducible assignments of spectra to cells

In [ ]:
# The mask contains six labelled mock cells; zero represents non-annotated tissue.
spatial = SPATIAL(nar_ids=np.array([0]), cut_off=20)
spatial.from_array(data["mask"])

fig, ax = plt.subplots(figsize=(5, 5))
image = ax.imshow(spatial.view, origin="lower")
fig.colorbar(image, ax=ax, label="Refactored region ID")
ax.set(title="Bundled mock cell mask", xlabel="x pixel", ylabel="y pixel")
plt.show()

In [ ]:
# Representative spectra and their sampling probabilities are bundled with the mask.
spectral = SPECTRAL(
    cell=data["cell_spectra"],
    nar=data["nar_spectra"],
    cell_distribution=data["cell_distribution"],
    nar_distribution=data["nar_distribution"],
)

fig, ax = plt.subplots(figsize=(8, 3))
for spectrum in spectral.cell:
    ax.plot(data["mz"], spectrum)
ax.plot(data["mz"], spectral.nar[0], "--", color="black", label="non-annotated region")
ax.set(xlabel="m/z", ylabel="mock intensity", title="Bundled representative spectra")
ax.legend()
plt.show()

In [ ]:
# MIX is used to assign spectra to spatial regions and construct the mixed observation.
mixed_data = MIX(spatial, spectral)
mixed_data.link()
mixed_data.downsample(scale=8, dtype="linear")

fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(mixed_data.mixed_block.sum(axis=2), origin="lower")
fig.colorbar(image, ax=ax, label="total mock ion intensity")
ax.set(title="Synthetic mixed image", xlabel="x bin", ylabel="y bin")
plt.show()

print("Overlap matrix:", mixed_data.overlap.shape)
print("Mixed data matrix:", mixed_data.mixed.shape)

In [ ]:
# Recover one spectrum per spatial region. There is no `unmix` module: use a solver class directly.
solver = NNLS(mixed_data.overlap, mixed_data.mixed)
solver.run()

relative_error = np.linalg.norm(solver.c - mixed_data.spectra, ord="fro") / np.linalg.norm(mixed_data.spectra, ord="fro")
print(f"Relative recovery error: {relative_error:.2e}")
assert relative_error < 1e-6